# LSTM Evaluation and Transfer Learning

This notebook evaluates the pretrained LSTM model on validation cities and on
München (zero-shot transfer). It then fine-tunes the prediction head using 7 days
of München data and compares the results.

**Prerequisites**: Run `04_lstm_model_training.ipynb` first to generate
`results/lstm/lstm_pretrained.pt` and `results/lstm/dataset_h3_multicidad.parquet`.

## 1. Configuration and Imports

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import h3
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_PATH = Path('../results/lstm/dataset_h3_multicidad.parquet')
MODEL_PATH = Path('../results/lstm/lstm_pretrained.pt')
RESULTS_DIR = Path('../results/lstm')
FIGURES_DIR = Path('../figures')

CITY_TEST = 'muenchen'
MIN_ACTIVE_PCT = 0.15
WINDOW_SIZE = 24
VAL_DAYS = 7
HIDDEN_SIZE  = 64
NUM_LAYERS   = 2
DROPOUT      = 0.2
BATCH_SIZE   = 512
N_FEATURES   = 6

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 10,
    'figure.dpi': 150,
})

## 2. Model and Dataset Definitions

In [ ]:
FEATURE_COLS = ['target_demanda', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend']


class SlidingWindowDataset(Dataset):
    def __init__(self, df_subset, T=24):
        self.T = T
        windows, targets, denorm_params = [], [], []
        for (city, cell), grp in df_subset.groupby(['city', 'h3_cell']):
            grp = grp.sort_values(['date', 'hour'])
            vals = grp[FEATURE_COLS].values.astype(np.float32)
            mu   = grp['mean'].iloc[0]
            sigma = grp['std'].iloc[0]
            for i in range(T, len(vals)):
                windows.append(vals[i-T:i])
                targets.append(vals[i, 0])
                denorm_params.append((mu, sigma))
        self.X = torch.tensor(np.array(windows), dtype=torch.float32)
        self.y = torch.tensor(np.array(targets),  dtype=torch.float32)
        self.denorm = denorm_params

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def denormalize(y_norm, denorm_params):
    result = np.zeros_like(y_norm)
    for i, (mu, sigma) in enumerate(denorm_params):
        result[i] = y_norm[i] * sigma + mu
    return np.clip(result, 0, None)


class DemandLSTM(nn.Module):
    def __init__(self, n_features, hidden_size, num_layers, dropout):
        super().__init__()
        self.encoder = nn.LSTM(
            input_size=n_features, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        out, _ = self.encoder(x)
        return self.head(out[:, -1, :]).squeeze(-1)

## 3. Data Loading and Preparation

In [ ]:
df = pd.read_parquet(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df.sort_values(['city', 'h3_cell', 'date', 'hour'], inplace=True)
df.reset_index(drop=True, inplace=True)

cell_activity = (
    df.groupby(['city', 'h3_cell'])['target_demanda']
    .apply(lambda x: (x > 0).mean())
    .reset_index(name='pct_active')
)
active_cells = cell_activity[cell_activity['pct_active'] >= MIN_ACTIVE_PCT]
df = df.merge(active_cells[['city', 'h3_cell']], on=['city', 'h3_cell'], how='inner')

cell_stats = df.groupby(['city', 'h3_cell'])['target_demanda'].agg(['mean', 'std']).reset_index()
cell_stats['std'] = cell_stats['std'].replace(0, 1)
df = df.merge(cell_stats, on=['city', 'h3_cell'], how='left')
df['demand_norm'] = (df['target_demanda'] - df['mean']) / df['std']
df['target_demanda_orig'] = df['target_demanda'].copy()
df['target_demanda'] = df['demand_norm']

df_pretrain = df[df['city'] != CITY_TEST].copy()
df_zeroshot = df[df['city'] == CITY_TEST].copy()

cutoff_dates = df_pretrain.groupby('city')['date'].max().apply(lambda d: d - pd.Timedelta(days=VAL_DAYS))
train_mask = df_pretrain.apply(lambda row: row['date'] <= cutoff_dates[row['city']], axis=1)
df_val = df_pretrain[~train_mask].copy()

val_ds = SlidingWindowDataset(df_val, T=WINDOW_SIZE)
test_ds = SlidingWindowDataset(df_zeroshot, T=WINDOW_SIZE)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f'Val windows:  {len(val_ds):,}')
print(f'Test windows: {len(test_ds):,}')

## 4. Loading the Pretrained Model

In [ ]:
model = DemandLSTM(N_FEATURES, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print('Pretrained model loaded.')

## 5. Evaluation Function

In [ ]:
def evaluate(loader, dataset, label, eval_model=None):
    m = eval_model if eval_model is not None else model
    m.eval()
    preds_norm, targets_norm = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            preds_norm.append(m(X_batch).cpu().numpy())
            targets_norm.append(y_batch.numpy())

    preds_norm   = np.concatenate(preds_norm)
    targets_norm = np.concatenate(targets_norm)
    preds_real   = denormalize(preds_norm, dataset.denorm)
    targets_real = denormalize(targets_norm, dataset.denorm)

    mae  = mean_absolute_error(targets_real, preds_real)
    rmse = np.sqrt(mean_squared_error(targets_real, preds_real))
    r2   = r2_score(targets_real, preds_real)

    denom = (np.abs(targets_real) + np.abs(preds_real)) / 2
    mask_s = denom > 0
    smape = np.mean(np.abs(targets_real[mask_s] - preds_real[mask_s]) / denom[mask_s]) * 100

    preds_rounded = np.round(preds_real)
    targets_rounded = np.round(targets_real)
    errors = np.abs(targets_rounded - preds_rounded)
    acc_at_0 = np.mean(errors == 0) * 100
    acc_at_1 = np.mean(errors <= 1) * 100

    print(f'[{label}]  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  sMAPE={smape:.2f}%')
    print(f'  Accuracy@0={acc_at_0:.1f}%  @1={acc_at_1:.1f}%')
    return {'label': label, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'sMAPE': smape,
            'Acc@0': acc_at_0, 'Acc@1': acc_at_1,
            'preds': preds_real, 'targets': targets_real}

## 6. Validation and Zero-Shot Results

In [ ]:
results_val  = evaluate(val_loader,  val_ds,  'Val (9 cities, last week)')
results_test = evaluate(test_loader, test_ds, 'Zero-shot Munich')

### Per-City Validation Metrics

In [ ]:
city_metrics = []
for city in sorted(df_val['city'].unique()):
    df_city = df_val[df_val['city'] == city]
    ds_city = SlidingWindowDataset(df_city, T=WINDOW_SIZE)
    if len(ds_city) == 0:
        continue
    loader_city = DataLoader(ds_city, batch_size=BATCH_SIZE, shuffle=False)
    m = evaluate(loader_city, ds_city, city)
    city_metrics.append(m)

city_metrics.append(results_test)

## 7. Comparative Table

In [ ]:
baseline = {'label': 'XGBoost (Milan)', 'MAE': 0.9247, 'RMSE': 1.4663, 'R2': 0.5660}

rows = [baseline] + [
    {'label': f'LSTM {m["label"]}', 'MAE': m['MAE'], 'RMSE': m['RMSE'],
     'R2': m['R2'], 'sMAPE': m.get('sMAPE', None)}
    for m in city_metrics
]

df_results = pd.DataFrame(rows).set_index('label').round(4)
print(df_results.to_string())

## 8. Time Series Visualisation — Milan

In [ ]:
df_milano_val = df_val[df_val['city'] == 'milano'].copy()
cell_demand = df_milano_val.groupby('h3_cell')['target_demanda_orig'].mean()
top_cell = cell_demand.idxmax()
print(f'Highest-demand H3 cell (Milan val): {top_cell} (mean={cell_demand[top_cell]:.2f})')

df_top = df_milano_val[df_milano_val['h3_cell'] == top_cell].copy()
ds_top = SlidingWindowDataset(df_top, T=WINDOW_SIZE)
loader_top = DataLoader(ds_top, batch_size=BATCH_SIZE, shuffle=False)

model.eval()
preds_n, tgts_n = [], []
with torch.no_grad():
    for X_b, y_b in loader_top:
        preds_n.append(model(X_b.to(DEVICE)).cpu().numpy())
        tgts_n.append(y_b.numpy())

preds_cell = denormalize(np.concatenate(preds_n), ds_top.denorm)
tgts_cell  = denormalize(np.concatenate(tgts_n), ds_top.denorm)

df_top_sorted = df_top.sort_values(['date', 'hour']).reset_index(drop=True)
df_plot = df_top_sorted.iloc[WINDOW_SIZE:].reset_index(drop=True)
df_plot['datetime'] = pd.to_datetime(df_plot['date']) + pd.to_timedelta(df_plot['hour'], unit='h')

fig, ax = plt.subplots(figsize=(14, 7), facecolor='white')
ax.plot(df_plot['datetime'].values, tgts_cell, color='#2196F3', linewidth=1.4,
        alpha=0.85, label='Observed demand', zorder=3)
ax.plot(df_plot['datetime'].values, preds_cell, color='#E65100', linewidth=1.4,
        alpha=0.85, linestyle='--', label='LSTM prediction', zorder=4)
ax.fill_between(df_plot['datetime'].values, tgts_cell, preds_cell,
                alpha=0.12, color='#FF7043', zorder=2)

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Hourly pickups', fontsize=11)
ax.set_title('Hourly Demand: Observed vs. Predicted — Highest-Demand H3 Cell (Milan)',
             fontsize=12, fontweight='bold', pad=10)
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=[6, 12, 18]))
fig.autofmt_xdate(rotation=30, ha='right')
ax.legend(loc='upper right', fontsize=10, framealpha=0.9, edgecolor='#cccccc', fancybox=False)
ax.grid(axis='both', alpha=0.25, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/lstm-timeseries-milan.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 9. Fine-Tuning on München

The encoder (LSTM layers) is frozen and only the prediction head is retrained
using 7 days of München data. This tests whether minimal local data can
improve the zero-shot baseline.

In [ ]:
FINETUNE_DAYS = 7

muen_dates     = sorted(df_zeroshot['date'].unique())
finetune_dates = muen_dates[:FINETUNE_DAYS]

df_muen_ft   = df_zeroshot[df_zeroshot['date'].isin(finetune_dates)]
df_muen_eval = df_zeroshot[~df_zeroshot['date'].isin(finetune_dates)]

ft_ds     = SlidingWindowDataset(df_muen_ft,   T=WINDOW_SIZE)
eval_ds   = SlidingWindowDataset(df_muen_eval, T=WINDOW_SIZE)
ft_loader = DataLoader(ft_ds,   batch_size=128, shuffle=True)
ev_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f'Fine-tuning: {len(df_muen_ft):,} rows ({FINETUNE_DAYS} days)')
print(f'Evaluation:  {len(df_muen_eval):,} rows')

In [ ]:
model_ft = DemandLSTM(N_FEATURES, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE)
model_ft.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

for param in model_ft.encoder.parameters():
    param.requires_grad = False

ft_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_ft.parameters()),
    lr=1e-4
)

criterion = nn.HuberLoss(delta=1.0)
FT_EPOCHS = 20

for epoch in range(1, FT_EPOCHS + 1):
    model_ft.train()
    ft_loss = 0.0
    for X_batch, y_batch in ft_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        ft_optimizer.zero_grad()
        pred = model_ft(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        ft_optimizer.step()
        ft_loss += loss.item() * len(y_batch)
    ft_loss /= len(ft_ds)
    if epoch % 5 == 0:
        print(f'FT Epoch {epoch:2d}/{FT_EPOCHS} | Loss: {ft_loss:.4f}')

torch.save(model_ft.state_dict(), RESULTS_DIR / 'lstm_finetuned_muenchen.pt')
print('Fine-tuned model saved.')

### Zero-Shot vs. Fine-Tuned Comparison

In [ ]:
results_ft = evaluate(ev_loader, eval_ds, 'Fine-tuned Munich (7 days FT)', eval_model=model_ft)

print('\nFinal comparison:')
rows_final = [
    {'Model': 'XGBoost Milan (baseline)',          'MAE': 0.9247, 'RMSE': 1.4663, 'R2': 0.5660},
    {'Model': 'LSTM zero-shot Munich',              'MAE': results_test['MAE'], 'RMSE': results_test['RMSE'], 'R2': results_test['R2']},
    {'Model': 'LSTM fine-tuned Munich (7 days)',     'MAE': results_ft['MAE'],  'RMSE': results_ft['RMSE'],  'R2': results_ft['R2']},
]
print(pd.DataFrame(rows_final).set_index('Model').round(4).to_string())

## 10. Time Series — München (Zero-Shot vs. Fine-Tuned)

In [ ]:
cell_demand_muen = df_muen_eval.groupby('h3_cell')['target_demanda_orig'].mean()
top_cell_muen = cell_demand_muen.idxmax()
print(f'Top H3 cell Munich: {top_cell_muen} (mean={cell_demand_muen[top_cell_muen]:.2f})')

df_top_muen = df_muen_eval[df_muen_eval['h3_cell'] == top_cell_muen].copy()

ds_top_muen = SlidingWindowDataset(df_top_muen, T=WINDOW_SIZE)
loader_top_muen = DataLoader(ds_top_muen, batch_size=BATCH_SIZE, shuffle=False)

preds_zs, preds_ft_cell, tgts_muen = [], [], []
model.eval()
model_ft.eval()
with torch.no_grad():
    for X_b, y_b in loader_top_muen:
        X_b = X_b.to(DEVICE)
        preds_zs.append(model(X_b).cpu().numpy())
        preds_ft_cell.append(model_ft(X_b).cpu().numpy())
        tgts_muen.append(y_b.numpy())

preds_zs = denormalize(np.concatenate(preds_zs), ds_top_muen.denorm)
preds_ft_cell = denormalize(np.concatenate(preds_ft_cell), ds_top_muen.denorm)
tgts_muen = denormalize(np.concatenate(tgts_muen), ds_top_muen.denorm)

df_top_muen_sorted = df_top_muen.sort_values(['date', 'hour']).reset_index(drop=True)
df_plot_muen = df_top_muen_sorted.iloc[WINDOW_SIZE:].reset_index(drop=True)
df_plot_muen['datetime'] = pd.to_datetime(df_plot_muen['date']) + pd.to_timedelta(df_plot_muen['hour'], unit='h')

df_plot_muen['pred_zs'] = preds_zs
df_plot_muen['pred_ft'] = preds_ft_cell
df_plot_muen['obs'] = tgts_muen

df_daily = df_plot_muen.groupby('date')[['obs', 'pred_zs', 'pred_ft']].mean().reset_index()
df_daily['date'] = pd.to_datetime(df_daily['date'])
df_daily = df_daily.iloc[:-1]

fig, ax = plt.subplots(figsize=(14, 7), facecolor='white')

ax.plot(df_daily['date'].values, df_daily['obs'], color='#2196F3',
        linewidth=2.0, alpha=0.9, marker='o', markersize=4, markerfacecolor='white', markeredgewidth=1.2,
        label='Observed demand', zorder=5)

ax.plot(df_daily['date'].values, df_daily['pred_zs'], color='#4CAF50',
        linewidth=1.8, alpha=0.85, linestyle='-.', marker='s', markersize=4, markerfacecolor='white', markeredgewidth=1.2,
        label='Zero-shot prediction', zorder=3)

ax.plot(df_daily['date'].values, df_daily['pred_ft'], color='#E65100',
        linewidth=1.8, alpha=0.85, linestyle='--', marker='^', markersize=4, markerfacecolor='white', markeredgewidth=1.2,
        label='Fine-tuned prediction', zorder=4)

ax.fill_between(df_daily['date'].values, df_daily['obs'], df_daily['pred_ft'],
                alpha=0.10, color='#FF7043', zorder=2)

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Mean daily pickups', fontsize=11)
ax.set_title('Daily Mean Demand: Observed vs. Zero-shot vs. Fine-tuned — Highest-Demand H3 Cell (München)',
             fontsize=12, fontweight='bold', pad=10)

ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

fig.autofmt_xdate(rotation=30, ha='right')
ax.legend(loc='upper right', fontsize=10, framealpha=0.9, edgecolor='#cccccc', fancybox=False)
ax.grid(axis='both', alpha=0.25, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/lstm-timesries-daily-munich.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 11. Fair Comparison: LSTM Predictions Aggregated to ACE Zones

The LSTM operates on H3 hexagonal cells while the XGBoost baseline uses ACE census
zones. For a fair comparison, LSTM predictions at the H3 level are spatially joined
to ACE zones and summed per (ACE, date, hour) slot. This produces metrics on the
same spatial granularity as the XGBoost model.

In [ ]:
ace_shp = gpd.read_file('../datasets/lombardia_shapefile/R03_11_WGS84.shp')
ace_shp = ace_shp.set_crs(epsg=32632).to_crs(epsg=4326)
ace_zones = ace_shp[
    (ace_shp['PRO_COM'] == 15146) & (ace_shp['ACE'] != 0)
].copy()
ace_zones = ace_zones.dissolve(by='ACE', as_index=False)[['ACE', 'geometry']]
print(f"ACE zones: {len(ace_zones)}")

h3_cells = df_milano_val['h3_cell'].unique()
h3_points = pd.DataFrame({
    'h3_cell': h3_cells,
    'lat': [h3.cell_to_latlng(c)[0] for c in h3_cells],
    'lon': [h3.cell_to_latlng(c)[1] for c in h3_cells],
})
gdf_h3 = gpd.GeoDataFrame(
    h3_points,
    geometry=gpd.points_from_xy(h3_points['lon'], h3_points['lat']),
    crs='EPSG:4326'
)

h3_to_ace = gpd.sjoin(gdf_h3, ace_zones, how='left', predicate='within')
h3_to_ace = h3_to_ace[['h3_cell', 'ACE']].dropna(subset=['ACE'])
h3_to_ace['ACE'] = h3_to_ace['ACE'].astype(int)
print(f"H3 cells matched to ACE: {len(h3_to_ace)} / {len(h3_cells)}")

In [ ]:
ds_milano = SlidingWindowDataset(df_milano_val, T=WINDOW_SIZE)
loader_milano = DataLoader(ds_milano, batch_size=BATCH_SIZE, shuffle=False)

model.eval()
preds_n, tgts_n = [], []
with torch.no_grad():
    for X_b, y_b in loader_milano:
        preds_n.append(model(X_b.to(DEVICE)).cpu().numpy())
        tgts_n.append(y_b.numpy())

preds_real = denormalize(np.concatenate(preds_n), ds_milano.denorm)
targets_real = denormalize(np.concatenate(tgts_n), ds_milano.denorm)

rows = []
for (city, cell), grp in df_milano_val.groupby(['city', 'h3_cell']):
    grp_sorted = grp.sort_values(['date', 'hour'])
    pred_rows = grp_sorted.iloc[WINDOW_SIZE:]
    rows.append(pred_rows[['h3_cell', 'date', 'hour']])

df_pred = pd.concat(rows, ignore_index=True)
df_pred['pred_lstm'] = preds_real
df_pred['actual'] = targets_real

df_pred = df_pred.merge(h3_to_ace, on='h3_cell', how='inner')

ace_agg = df_pred.groupby(['ACE', 'date', 'hour']).agg(
    pred_lstm=('pred_lstm', 'sum'),
    actual=('actual', 'sum')
).reset_index()

mae = mean_absolute_error(ace_agg['actual'], ace_agg['pred_lstm'])
rmse = np.sqrt(mean_squared_error(ace_agg['actual'], ace_agg['pred_lstm']))
r2 = r2_score(ace_agg['actual'], ace_agg['pred_lstm'])

preds_round = np.round(ace_agg['pred_lstm'].values)
tgts_round = np.round(ace_agg['actual'].values)
errors = np.abs(preds_round - tgts_round)
acc0 = np.mean(errors == 0) * 100
acc1 = np.mean(errors <= 1) * 100

print(f"LSTM aggregated to ACE zones (Milan validation)")
print(f"MAE:   {mae:.4f}")
print(f"RMSE:  {rmse:.4f}")
print(f"R2:    {r2:.4f}")
print(f"Acc@0: {acc0:.1f}%")
print(f"Acc@1: {acc1:.1f}%")

## 12. Saving Predictions

In [ ]:
preds_all_n, tgts_all_n = [], []
model.eval()
with torch.no_grad():
    for X_b, y_b in test_loader:
        preds_all_n.append(model(X_b.to(DEVICE)).cpu().numpy())
        tgts_all_n.append(y_b.numpy())

preds_all = denormalize(np.concatenate(preds_all_n), test_ds.denorm)
tgts_all = denormalize(np.concatenate(tgts_all_n), test_ds.denorm)

pred_df = pd.DataFrame({
    'actual': tgts_all,
    'predicted_zeroshot': preds_all,
})
pred_df.to_csv(RESULTS_DIR / 'lstm_predictions_munich.csv', index=False)
print(f"Saved {len(pred_df):,} predictions to results/lstm/lstm_predictions_munich.csv")